# Nine-class EG-VAN: gated Colab training handoff

Choose **Runtime → Change runtime type → GPU** with a compatible CUDA-enabled PyTorch runtime. This notebook does not train by default. It requires a real approved nine-class cohort, frozen protected test, and original images in the mounted project tree. It calls the repository's Stage 6 runner; it does not contain a second training loop.

Do not run final test evaluation while selecting a checkpoint. See `docs/NINE_CLASS_COLAB_TRAINING.md`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Locate the project and install pinned runtime dependencies

Place the current project code, approved manifests, and original data under one stable Drive project root. The default path below is an example. The notebook will stop if it is absent. The code does not download datasets, pretrained weights, or clone a repository.

In [ ]:
from pathlib import Path
import os, sys, subprocess, platform, json, shlex
PROJECT_ROOT = Path('/content/drive/MyDrive/EG-VAN').resolve()
if not (PROJECT_ROOT / 'src/nine_class_training.py').is_file():
    raise FileNotFoundError(f'Current EG-VAN project code not found: {PROJECT_ROOT}')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
requirements = PROJECT_ROOT / 'config/requirements_colab_2026_07.txt'
if not requirements.is_file():
    raise FileNotFoundError(requirements)
print('Project root:', PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)], check=True)

In [ ]:
import torch, torchvision, numpy as np, cv2
from nine_class_stage4 import CLASS_NAMES
from nine_class_colab import (validate_colab_config, validate_colab_runtime, validate_cohort,
                              gpu_memory_plan, training_command, evaluation_command)
from nine_class_dataset import NineClassDataset, preprocessing_for_training
EXPECTED_LABELS = ('AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'VASC', 'BENIGN', 'MALIGNANT')
if sys.version_info < (3, 10):
    raise RuntimeError('Python 3.10 or newer is required.')
runtime = validate_colab_runtime(torch, torchvision)
with torch.autocast(device_type='cuda', dtype=torch.float16):
    half_probe = torch.ones((8, 8), device='cuda') @ torch.ones((8, 8), device='cuda')
torch.cuda.synchronize()
if not bool(torch.isfinite(half_probe).all().item()):
    raise RuntimeError('CUDA FP16 execution probe failed.')
if tuple(CLASS_NAMES) != EXPECTED_LABELS:
    raise RuntimeError('Nine-class label order mismatch.')
print('GPU:', torch.cuda.get_device_name(0), 'CUDA:', torch.version.cuda)
print('Python:', platform.python_version(), 'torch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('numpy:', np.__version__, 'OpenCV:', cv2.__version__)
subprocess.run(['nvidia-smi'], check=False)

## Approved cohort gate

Replace the two example paths only after a real eligibility decision. The Stage 6 validator checks the approval record, image hashes, labels, overlap groups, protected publisher test, and all-nine train/validation/test support. An absent or unapproved cohort stops here before image loading or model construction.

In [ ]:
CONFIG = PROJECT_ROOT / 'config/nine_class_training_t4_v1.json'
INVENTORY = PROJECT_ROOT / 'analysis/nine_class_stage4/nine_class_source_manifest.csv'
APPROVED_SPLIT = PROJECT_ROOT / 'analysis/approved_nine_class/approved_split.csv'  # replace after approval
APPROVAL = PROJECT_ROOT / 'analysis/approved_nine_class/approval.json'              # replace after approval
RUNS_ROOT = PROJECT_ROOT / 'analysis/nine_class_stage6/runs'  # persistent Drive location
for name, path in [('configuration', CONFIG), ('inventory', INVENTORY),
                   ('approved split', APPROVED_SPLIT), ('approval', APPROVAL)]:
    if not path.is_file():
        raise FileNotFoundError(f'{name} is missing; dataset approval is required before GPU training: {path}')
plan = validate_colab_config(CONFIG)
print('Preprocessing:', plan['config'].get('preprocessing_mode', 'stage4_full_v1'))
print('Preprocessing SHA-256:', plan['preprocess_config_sha256'])
print('Microbatch / accumulation / nominal effective batch:',
      plan['microbatch_size'], plan['accumulation_steps'], plan['nominal_effective_batch_size'])
cohort = validate_cohort(PROJECT_ROOT, APPROVED_SPLIT, APPROVAL, INVENTORY)
print('Approved split counts:', cohort['provenance']['counts'])

In [ ]:
free_bytes, total_bytes = torch.cuda.mem_get_info()
memory = gpu_memory_plan(total_bytes, free_bytes, plan['config'])
print(json.dumps(memory, indent=2))
if free_bytes < 4 * 2**30:
    raise RuntimeError('Less than 4 GiB free GPU memory. Stop and review resource plan; no batch size is guaranteed.')
# The loader smoke test uses approved validation data only; protected test stays untouched.
preprocess = preprocessing_for_training(plan['config'])
validation = NineClassDataset(PROJECT_ROOT, APPROVED_SPLIT, 'validation', preprocess)
image1, label1 = validation[0]
image2, label2 = validation[0]
if image1.shape != (3, 384, 384) or not torch.equal(image1, image2) or not 0 <= label1 < 9 or label1 != label2:
    raise RuntimeError('Approved-data deterministic validation smoke test failed.')
print('Approved validation image smoke test passed; nine-class index:', label1)

## Batch-one GPU model smoke test

This reads one **approved validation** image and builds random-initialized EG-VAN. It performs inference only. If CUDA runs out of memory, stop; do not start training. A batch-one inference pass is not a training peak-memory measurement.

In [ ]:
from models.egvan import build_nine_class_egvan
try:
    model = build_nine_class_egvan(fusion_channels=int(plan['config']['fusion_channels'])).cuda().eval()
    with torch.inference_mode():
        logits = model(image1.unsqueeze(0).cuda())
    if tuple(logits.shape) != (1, 9) or not torch.isfinite(logits).all():
        raise RuntimeError('Nine-logit GPU smoke test failed.')
    print('Batch-one GPU forward passed:', tuple(logits.shape))
finally:
    if 'model' in globals():
        del model
    if 'logits' in globals():
        del logits
    torch.cuda.empty_cache()

## Recheck the production CLI gate and preserve environment details

The exact validation command is printed. The runner's `--validate-only` path does not construct the model or train. The environment record is written to persistent Drive only after the cohort gate passes.

In [ ]:
validate_cmd = training_command(PROJECT_ROOT, CONFIG, APPROVED_SPLIT, APPROVAL, INVENTORY,
                                RUNS_ROOT, validate_only=True)
print('VALIDATION COMMAND:', shlex.join(validate_cmd))
subprocess.run(validate_cmd, check=True)
from datetime import datetime, timezone
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
env_record = RUNS_ROOT / f'colab_environment_{stamp}.json'
env_record.write_text(json.dumps({
    'python': platform.python_version(), 'torch': torch.__version__,
    'torchvision': torchvision.__version__, 'numpy': np.__version__,
    'opencv': cv2.__version__, 'gpu': torch.cuda.get_device_name(0),
    'config': str(CONFIG), 'preprocess_config_sha256': plan['preprocess_config_sha256'],
    'approved_split': str(APPROVED_SPLIT), 'approval': str(APPROVAL)}, indent=2) + '\n')
print('Persistent environment record:', env_record)

## Full training: explicit opt-in

Set RUN_TRAINING=True only after all prior cells pass and the researcher authorizes the actual approved cohort. The T4 configuration uses microbatch 8 with four gradient accumulation steps (nominal effective batch 32) and frozen BatchNorm running statistics. The final partial update uses its actual sample count. An OOM stops the run; inspect the last completed checkpoint. Changing microbatch or accumulation changes the configuration hash and requires a new run, not resume into the old run.

In [ ]:
RUN_TRAINING = False
train_cmd = training_command(PROJECT_ROOT, CONFIG, APPROVED_SPLIT, APPROVAL, INVENTORY, RUNS_ROOT)
print('TRAINING COMMAND:', shlex.join(train_cmd))
if RUN_TRAINING:
    validate_cohort(PROJECT_ROOT, APPROVED_SPLIT, APPROVAL, INVENTORY)
    if not torch.cuda.is_available():
        raise RuntimeError('GPU disappeared before training.')
    try:
        subprocess.run(train_cmd, check=True)
    except subprocess.CalledProcessError as error:
        raise RuntimeError('Training stopped. If CUDA OOM occurred, inspect the last complete last.pt; choose a smaller microbatch in a NEW versioned config and run after fresh preflight. Do not silently reuse the old checkpoint.') from error
else:
    print('Training remains disabled (RUN_TRAINING=False).')

## Safe resume after disconnection

The Stage 6 runner writes `last.pt`, `best.pt`, `history.csv`, and run provenance in the persistent run directory. Set the existing `last.pt` path below, then opt in. Resume rejects changed config, cohort and preprocessing hashes. `best.pt` is for evaluation, not resume.

In [ ]:
RUN_RESUME = False
RESUME_CHECKPOINT = RUNS_ROOT / 'REPLACE_WITH_REAL_RUN_DIR/last.pt'
if RUN_RESUME:
    validate_cohort(PROJECT_ROOT, APPROVED_SPLIT, APPROVAL, INVENTORY)
    if not torch.cuda.is_available():
        raise RuntimeError('GPU unavailable for resume.')
    resume_cmd = training_command(PROJECT_ROOT, CONFIG, APPROVED_SPLIT, APPROVAL, INVENTORY,
                                  RUNS_ROOT, resume=RESUME_CHECKPOINT)
    print('RESUME COMMAND:', shlex.join(resume_cmd))
    subprocess.run(resume_cmd, check=True)
else:
    print('Resume disabled; set RUN_RESUME=True only with a matching persistent last.pt.')

## Evaluation: validation first, protected test only at the declared final assessment

These commands call the separate Stage 6 evaluator. No test result is generated by default. Use a new output directory for each evaluation. The final test requires a deliberate second flag and the runner's `--final-test` gate.

In [ ]:
RUN_VALIDATION_EVAL = False
RUN_FINAL_TEST = False
BEST_CHECKPOINT = RUNS_ROOT / 'REPLACE_WITH_REAL_RUN_DIR/best.pt'
VALIDATION_OUTPUT = RUNS_ROOT / 'REPLACE_WITH_NEW_VALIDATION_OUTPUT_DIR'
TEST_OUTPUT = RUNS_ROOT / 'REPLACE_WITH_NEW_FINAL_TEST_OUTPUT_DIR'
if RUN_VALIDATION_EVAL:
    validate_cohort(PROJECT_ROOT, APPROVED_SPLIT, APPROVAL, INVENTORY)
    cmd = evaluation_command(PROJECT_ROOT, APPROVED_SPLIT, APPROVAL, INVENTORY,
                             BEST_CHECKPOINT, VALIDATION_OUTPUT)
    print('VALIDATION EVALUATION COMMAND:', shlex.join(cmd))
    subprocess.run(cmd, check=True)
if RUN_FINAL_TEST:
    validate_cohort(PROJECT_ROOT, APPROVED_SPLIT, APPROVAL, INVENTORY)
    cmd = evaluation_command(PROJECT_ROOT, APPROVED_SPLIT, APPROVAL, INVENTORY,
                             BEST_CHECKPOINT, TEST_OUTPUT, final_test=True)
    print('FINAL TEST COMMAND:', shlex.join(cmd))
    subprocess.run(cmd, check=True)
if not RUN_VALIDATION_EVAL and not RUN_FINAL_TEST:
    print('Evaluation disabled; no validation or protected-test predictions generated.')